In [ ]:
! pip install -q "datasets<4.0.0"  transformers seqeval

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

import numpy as np
from sklearn.metrics import f1_score as sklearn_f1

import torch
from sklearn.model_selection import train_test_split
from seqeval.metrics import f1_score

In [ ]:
dataset = load_dataset("jakartaresearch/semeval-absa", "restaurant")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'aspects', 'category'],
        num_rows: 3044
    })
    validation: Dataset({
        features: ['id', 'text', 'aspects', 'category'],
        num_rows: 800
    })
})


In [ ]:
aspect_labels = ["O", "B-ASP", "I-ASP"]
tag2id = {l: i for i, l in enumerate(aspect_labels)}
id2tag = {i: l for i, l in enumerate(aspect_labels)}

In [ ]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(aspect_labels),
    id2label=id2tag,
    label2id=tag2id
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_and_align_labels(batch):
    tokenized_inputs = tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding=False,
        return_offsets_mapping=True
    )

    all_labels = []

    for i in range(len(batch["text"])):
        raw_text = batch["text"][i]
        offsets = tokenized_inputs["offset_mapping"][i]
        aspect_dict = batch["aspects"][i]

        start_spans = aspect_dict.get("from", [])
        end_spans = aspect_dict.get("to", [])

        # Sắp xếp các span theo thứ tự xuất hiện trong câu
        valid_spans = sorted(
            [(s, e) for s, e in zip(start_spans, end_spans) if s < e],
            key=lambda x: x[0]
        )

        label_ids = []
        current_active_span = None
        prev_token_end = 0

        seq_ids = tokenized_inputs.sequence_ids(batch_index=i)

        for idx, (off_start, off_end) in enumerate(offsets):
            # 1. Special tokens ([CLS], [SEP]) -> -100
            if seq_ids[idx] is None or off_start == off_end:
                label_ids.append(-100)
                continue

            # 2. Tìm aspect span bao trùm token
            matched_span = None
            for asp_start, asp_end in valid_spans:
                if max(off_start, asp_start) < min(off_end, asp_end):
                    matched_span = (asp_start, asp_end)
                    break

            # 3. Gán nhãn
            if matched_span is None:
                label_ids.append(tag2id["O"])
                current_active_span = None
            else:
                # TRƯỜNG HỢP A: Bắt đầu một aspect term mới
                if current_active_span != matched_span:
                    label_ids.append(tag2id["B-ASP"])
                    current_active_span = matched_span

                # TRƯỜNG HỢP B: Vẫn nằm trong cùng aspect term đó
                else:
                    # Kiểm tra xem đây có phải là TỪ MỚI hay không:
                    # - Có khoảng trắng trước token (BERT)
                    # - HOẶC token bắt đầu bằng ký tự khoảng trắng (RoBERTa/DeBERTa)
                    is_new_word = (off_start > prev_token_end) or (
                        off_start < len(raw_text) and raw_text[off_start].isspace()
                    )

                    if is_new_word:
                        label_ids.append(tag2id["I-ASP"])  # 'with', 'sausage', 'chicken'
                    else:
                        label_ids.append(-100)  # Subwords: 'rec', 'hie', 'te'

            prev_token_end = off_end

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    tokenized_inputs.pop("offset_mapping")
    return tokenized_inputs

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/3044 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
sample = tokenized_dataset["train"][7]
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
for t, l in zip(tokens, sample["labels"]):
    print(f"{t:15} {l}")

<s>             -100
▁Our            0
▁agreed         0
▁favorite       0
▁is             0
▁the            0
▁or             1
re              -100
chie            -100
te              -100
▁with           2
▁sa             2
usage           -100
▁and            2
▁chicken        2
▁(              0
usu             0
ally            0
▁the            0
▁wait           1
ers             -100
▁are            0
▁kind           0
▁enough         0
▁to             0
▁split          0
▁the            0
▁di             1
sh              -100
▁in             0
▁half           0
▁so             0
▁you            0
▁get            0
▁to             0
▁sample         0
▁both           0
▁me             1
ats             -100
).              0
</s>            -100


In [ ]:
# Split train-test
train_val_dataset = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = train_val_dataset["train"]
eval_dataset = train_val_dataset["test"]
test_dataset =  tokenized_dataset["validation"]

In [ ]:
import numpy as np

def label_from_predictions(predictions, labels):
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return true_predictions, true_labels

In [ ]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    true_predictions, true_labels = label_from_predictions(predictions, labels)

    return {
        "f1": f1_score(true_labels, true_predictions),
    }

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir='/content/transformer_extractor',
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    lr_scheduler_type="linear",

    fp16=False,  # BẮT BUỘC tắt FP16 để tránh lỗi NaN
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),

    logging_strategy='epoch',
    logging_first_step=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,

    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.311937,0.073914,0.802057
2,0.068899,0.062313,0.853872


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# save model
trainer.save_model('/content/lstm_ner_final')

# Step 2:

In [ ]:
def prepare_for_sentiment(batch):
    batch_text = []
    batch_term = []
    batch_sentiment = []

    texts = batch["text"]
    aspects = batch["aspects"]
    num_examples = len(texts)

    for i in range(num_examples):
        text = texts[i]
        aspect = aspects[i]
        polarities = aspect["polarity"]
        terms = aspect["term"]

        if terms == [""] or term = "":
            continue

        for term, polarity in zip(terms, polarities):
            batch_text.append(text)
            batch_term.append(term)
            batch_sentiment.append(polarity)

    return {
        "text": batch_text,
        "term": batch_term,
        "polarity": batch_sentiment
    }

sentiment_dataset = dataset.map(
    prepare_for_sentiment,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/3044 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
labels = ['conflict', 'negative', 'neutral', 'positive']
id2label = {i : v for i, v in enumerate(labels)}
label2id = {v : i for i, v in enumerate(labels)}

In [ ]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize(batch):
    tokenized_inputs = tokenizer(
        batch["text"],
        batch["term"],
        truncation=True,
        padding=False
    )
    tokenized_inputs["labels"] = [label2id[label] for label in batch["polarity"]]

    return tokenized_inputs

tokenized_dataset = sentiment_dataset.map(
    tokenize,
    batched=True,
    remove_columns=sentiment_dataset["train"].column_names
)

Map:   0%|          | 0/3699 [00:00<?, ? examples/s]

Map:   0%|          | 0/1134 [00:00<?, ? examples/s]

In [ ]:
train_val_split = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)
train_dataset = train_val_split["train"]
eval_dataset = train_val_split["test"]
test_dataset = tokenized_dataset.get("validation")

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    return {
        "f1" : sklearn_f1(predictions, labels, average="macro")
    }


In [ ]:
training_args = TrainingArguments(
    output_dir='/content/sentiment_classifier',
    num_train_epochs=5,                 # Phân loại câu chỉ cần 4-5 epochs
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,      # Eval có thể để batch size to hơn để chạy nhanh
    warmup_steps=30,                   # Warmup 10% tổng số bước
    lr_scheduler_type="linear",

    fp16=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),

    logging_strategy='epoch',
    logging_first_step=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,

    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',   # Lưu checkpoint có Macro-F1 cao nhất
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Bắt đầu train
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,1.042890,0.761761,0.368285
2,0.727734,0.700233,0.468173


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# save model
trainer.save_model('/content/lstm_ner_final')

In [ ]:
import torch
import numpy as np

def extract_aspects_from_bio(text, tokens, pred_labels, id2tag):
    """
    Hàm ghép các token B-ASP và I-ASP thành từ/cụm từ hoàn chỉnh.
    """
    aspects = []
    current_aspect = []

    for token, label_id in zip(tokens, pred_labels):
        if label_id == -100:
            continue
        tag = id2tag[label_id]

        if tag == "B-ASP":
            if current_aspect:
                aspects.append(tokenizer_m1.convert_tokens_to_string(current_aspect).strip())
                current_aspect = []
            current_aspect.append(token)
        elif tag == "I-ASP":
            if current_aspect:
                current_aspect.append(token)
        else:  # 'O'
            if current_aspect:
                aspects.append(tokenizer_m1.convert_tokens_to_string(current_aspect).strip())
                current_aspect = []

    if current_aspect:
        aspects.append(tokenizer_m1.convert_tokens_to_string(current_aspect).strip())

    return aspects


def evaluate_end_to_end(model_ate, tokenizer_ate, model_asc, tokenizer_asc, test_dataset, id2polarity):
    model_ate.eval()
    model_asc.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_ate.to(device)
    model_asc.to(device)

    total_gold_count = 0
    total_pred_count = 0
    true_positives = 0

    with torch.no_grad():
        for sample in test_dataset:
            text = sample["text"]

            # 1. LẤY NHÃN THỰC TẾ (GOLD)
            # Dạng: set([("food", "positive"), ("service", "negative")])
            gold_terms = sample["aspects"]["term"]
            gold_polarities = sample["aspects"]["polarity"]
            gold_pairs = set(zip(gold_terms, gold_polarities))

            # Loại bỏ các span rỗng (nếu có trong dataset)
            gold_pairs = {p for p in gold_pairs if p[0].strip() != ""}
            total_gold_count += len(gold_pairs)

            # 2. CHẠY MODEL 1 (TRÍCH XUẤT ASPECT)
            inputs_m1 = tokenizer_ate(text, return_tensors="pt", truncation=True).to(device)
            logits_m1 = model_ate(**inputs_m1).logits
            preds_m1 = torch.argmax(logits_m1, dim=-1)[0].cpu().tolist()
            tokens_m1 = tokenizer_ate.convert_ids_to_tokens(inputs_m1["input_ids"][0])

            # Trích xuất các aspect tìm được
            pred_aspects = extract_aspects_from_bio(text, tokens_m1, preds_m1, id2tag)

            # 3. CHẠY MODEL 2 (DỰ ĐOÁN SENTIMENT CHO CÁC ASPECT VỪA TÌM ĐƯỢC)
            pred_pairs = set()
            for asp in pred_aspects:
                if not asp.strip():
                    continue

                # Format đầu vào chuẩn cho Model 2: sentence + [SEP] + aspect
                inputs_m2 = tokenizer_asc(text, asp, return_tensors="pt", truncation=True).to(device)
                logits_m2 = model_asc(**inputs_m2).logits
                pred_polarity_id = torch.argmax(logits_m2, dim=-1).item()
                pred_polarity = id2polarity[pred_polarity_id]

                pred_pairs.add((asp.lower(), pred_polarity.lower()))

            # Chuẩn hoá chữ thường để so khớp không phân biệt hoa/thường
            gold_pairs_lower = {(term.lower(), pol.lower()) for term, pol in gold_pairs}

            total_pred_count += len(pred_pairs)

            # Đếm số cặp khớp cả Aspect Term VÀ Polarity
            tp = len(pred_pairs.intersection(gold_pairs_lower))
            true_positives += tp

    # 4. TÍNH CHỈ SỐ CUỐI CÙNG
    precision = true_positives / total_pred_count if total_pred_count > 0 else 0.0
    recall = true_positives / total_gold_count if total_gold_count > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print("=" * 40)
    print("      END-TO-END PIPELINE EVALUATION")
    print("=" * 40)
    print(f"True Positives (TP): {true_positives}")
    print(f"Total Predicted:    {total_pred_count}")
    print(f"Total Ground Truth: {total_gold_count}")
    print("-" * 40)
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall:    {recall * 100:.2f}%")
    print(f"Micro-F1:  {f1 * 100:.2f}%")
    print("=" * 40)

    return {"precision": precision, "recall": recall, "f1": f1}